In [2]:
import json
import random
from collections import Counter
from pathlib import Path
from urllib.request import urlopen


manifest = Path("rps/rock-paper-scissors.ndjson")
if not manifest.exists():
    manifest = Path("PVA2/rps/rock-paper-scissors.ndjson")

with manifest.open() as f:
    records = [json.loads(line) for line in f if line.strip()]

metadata = next(r for r in records if r["type"] == "dataset")
class_names = metadata["class_names"]

images = [r for r in records if r["type"] == "image"]
random.Random(42).shuffle(images)

output_dir = manifest.parent / "classification"
limits = {"train": 200, "val": 30, "test": 30}
counts = Counter()

for record in images:
    boxes = record.get("annotations", {}).get("boxes", [])
    class_ids = {int(box[0]) for box in boxes}

    # Skip unlabelled images and images containing different classes.
    if len(class_ids) != 1:
        continue

    split = record["split"]
    if split not in limits:
        continue

    class_id = next(iter(class_ids))
    label = class_names[str(class_id)]
    key = (split, label)

    if counts[key] >= limits[split]:
        continue

    folder = output_dir / split / label
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / Path(record["file"]).name

    if not destination.exists():
        try:
            with urlopen(record["url"], timeout=30) as response:
                data = response.read()
            destination.write_bytes(data)
        except Exception as error:
            print(f"Download failed: {destination.name}: {type(error).__name__}")
            continue

    counts[key] += 1

for (split, label), count in sorted(counts.items()):
    print(f"{split}/{label}: {count} images")

print("Saved to:", output_dir.resolve())


test/Paper: 30 images
test/Rock: 30 images
test/Scissors: 30 images
train/Paper: 200 images
train/Rock: 200 images
train/Scissors: 200 images
val/Paper: 30 images
val/Rock: 30 images
val/Scissors: 30 images
Saved to: /home/ladina/Documents/git/FFHS/DeepLCodingKatas/PVA2/rps/classification


In [5]:
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

base_dir = Path.cwd()
if not (base_dir / "rps").is_dir():
    base_dir = base_dir / "PVA2"

model_dir = base_dir / "rps" / "model"

model = tf.keras.models.load_model(
    model_dir / "keras_model.h5",
    compile=False,
)

# Lines normally look like "0 Paper".
labels = [
    line.strip()
    for line in (model_dir / "labels.txt").read_text().splitlines()
    if line.strip()
]


print("Classes:", labels)
print("Input shape:", model.input_shape)


TypeError: Error when deserializing class 'DepthwiseConv2D' using config={'name': 'expanded_conv_depthwise', 'trainable': True, 'dtype': 'float32', 'kernel_size': [3, 3], 'strides': [1, 1], 'padding': 'same', 'data_format': 'channels_last', 'dilation_rate': [1, 1], 'groups': 1, 'activation': 'linear', 'use_bias': False, 'bias_initializer': {'class_name': 'Zeros', 'config': {}}, 'bias_regularizer': None, 'activity_regularizer': None, 'bias_constraint': None, 'depth_multiplier': 1, 'depthwise_initializer': {'class_name': 'VarianceScaling', 'config': {'scale': 1, 'mode': 'fan_avg', 'distribution': 'uniform', 'seed': None}}, 'depthwise_regularizer': None, 'depthwise_constraint': None}.

Exception encountered: Unrecognized keyword arguments passed to DepthwiseConv2D: {'groups': 1}

In [4]:
def predict_gesture(image_path):
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")

    # Resize and crop to the model's expected dimensions.
    height, width = model.input_shape[1:3]
    image = ImageOps.fit(
        image,
        (width, height),
        method=Image.Resampling.LANCZOS,
    )

    pixels = np.asarray(image, dtype=np.float32)
    normalized = pixels / 127.5 - 1.0

    # Add the batch dimension: (1, height, width, 3).
    batch = normalized[np.newaxis, ...]

    scores = model.predict(batch, verbose=0)[0]
    best_index = int(np.argmax(scores))

    plt.imshow(image)
    plt.title(f"{labels[best_index]}: {scores[best_index]:.1%}")
    plt.axis("off")
    plt.show()

    return dict(zip(labels, scores.tolist()))


In [3]:
test_dir = base_dir / "rps" / "classification" / "test" / "Rock"
test_image = next(test_dir.glob("*.jpg"))

scores = predict_gesture(test_image)

for label, score in scores.items():
    print(f"{label}: {score:.1%}")


NameError: name 'model' is not defined